In [ ]:
%pip install -q wfdb matplotlib scikit-learn pandas

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

In [ ]:
# --- Signal processing ---
def bandpass_filter(signal, lowcut=20, highcut=450, fs= 1000, order=4):
     nyq = 0.5 * fs
     low = lowcut / nyq
     high = highcut / nyq
     b, a = butter(order, [low,high], btype= 'band')
     return filtfilt(b, a, signal)

In [ ]:
def extract_features(segment):
    return [
        np.mean(np.abs(segment)),    #MAV
        np.sqrt(np.mean(segment**2)), #RMS
        np.var(segment),
        ((segment[:-1] * segment[1:]) < 0).sum() #Zero crossings
    ]

In [ ]:
classes = {
    "emg_healthy" : 0,
    "emg_myopathy" : 1,
    "emg_neuropathy" : 2
}
class_names = ["Healthy", "Myopathy", "Neuropathy"]

X_train, X_test, y_train, y_test = [], [], [], []

for record_name, label in classes.items():
    record = wfdb.rdrecord(record_name, pn_dir="emgdb/1.0.0")
    fs = record.fs
    window_size = round(0.2 * fs)
    step_size = window_size // 2
    signals = record.p_signal
    split = int(0.8 * len(signals))
    for part, features_out, labels_out in [
        (signals[:split], X_train, y_train),
        (signals[split:], X_test, y_test),
    ]:
        signals_filt = np.array([bandpass_filter(ch, fs=fs) for ch in part.T]).T

        for start in range(0, signals_filt.shape[0] - window_size + 1, step_size):
            window = signals_filt[start:start+window_size, :]
            features = []
            for ch in range(window.shape [1]):
                features.extend(extract_features(window[:, ch]))
            features_out.append(features)
            labels_out.append(label)

X_train, X_test = np.asarray(X_train), np.asarray(X_test)
y_train, y_test = np.asarray(y_train), np.asarray(y_test)
X = np.concatenate([X_train, X_test])
y = np.concatenate([y_train, y_test])

In [ ]:
# Convert lists to numpy arrays
X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

# Check class distribution
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {class_names[u]}: {c} samples ({c/len(y)*100:.2f}%)")


In [ ]:
print("Train set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc as curve_area
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import pandas as pd

# Number of classes
n_classes = len(np.unique(y))

results = []

for name, model in models.items():
    print("="*60)
    print(f"Model: {name}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", model)
    ])

    # Train
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # Classification Report
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred, labels=pipe.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    # Scores
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    # AUC (only if predict_proba is available)
    if hasattr(pipe.named_steps["clf"], "predict_proba"):
        y_prob = pipe.predict_proba(X_test)
        y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
        auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr", average="macro")
    else:
        auc = None

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = curve_area(fpr[i], tpr[i])

    # Plot all ROC curves
    plt.figure(figsize=(6, 6))
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i],
                 label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {name}")
    plt.legend(loc="lower right")
    plt.show()

    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-macro": f1,
        "AUC-macro": auc
    })

# Convert to DataFrame for easy comparison
df_results = pd.DataFrame(results)
print("\n=== Summary of Model Performance ===")
display(df_results)
